In [ ]:
from glob import glob
from scipy.io import loadmat, savemat
import os, re, json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm

file_path = "/mnt/DATA/Motion_fMRI/Datasets/eso245/"
file_name = "eso245_cra_strin_*.mat"
field_name = "subj_tcs"
to_throw = [
    1,
    12,
    27,
    36,
    46,
    60,
    167,
    169,
    180,
    185,
    187,
    210,
    215,
    219,
    229,
]  # [187, 215, 219, 36, 27, 46, 169, 1, 167, 180, 210, 12, 185, 97, 229, 60]

In [ ]:
A = sorted([187, 215, 219, 36, 27, 46, 169, 1, 167, 180, 210, 12, 185, 97, 229, 60])
B = sorted([187, 60, 185, 229, 180, 210, 12, 167, 1, 36, 169, 46, 27, 215, 219])
print(A, B, sep="\n")

In [ ]:
with open("../NonLinearityResultsNew/eso245_cra_strin_ValidSubjects.json") as fp:
    old_goods = json.load(fp)
len(old_goods), len(to_throw)

In [ ]:
files = {}
bad_guys = []
goods = np.full(245, True)
goods[to_throw] = False
to_throw_binary = np.zeros(245, dtype="int64")
to_throw_binary[to_throw] = 1
adjust = np.cumsum(to_throw_binary)[goods]
for file in glob(os.path.join(file_path, file_name)):
    findings = re.findall(r"_(\d+)\.mat", file)
    files[int(findings[0])] = file
for reg in sorted(files.keys()):
    file = files[reg]
    mat = loadmat(file)[field_name][:, :, goods]
    failed = np.sum(np.isclose(mat, 0), 0) == 400
    fail_per_reg = np.sum(failed, 1)
    # keep_reg = np.full(mat.shape[1], True)
    # if np.count_nonzero(fail_per_reg)<=len(to_throw):
    #     keep_reg[np.where(fail_per_reg>0)]=False
    # else:
    #     lower_throw = fail_per_reg[np.argsort(fail_per_reg)[-len(to_throw)]]
    #     keep_reg[np.where(fail_per_reg>=lower_throw)]=False
    # failed = failed[keep_reg,:]
    fail_per_sub = np.sum(failed, 0)
    full_good_sub = np.where(fail_per_sub == 0)
    full_failers = np.sum(fail_per_sub == mat.shape[1])
    fail_per_reg = np.sum(failed, 1)
    full_good_reg = np.where(fail_per_reg <= full_failers)[0]
    failed = failed[:, fail_per_sub > 0][fail_per_reg > full_failers, :]
    sub_ord = np.argsort(np.sum(failed, 0))
    reg_ord = np.argsort(np.sum(failed, 1))[::-1]
    print(failed.shape, mat.shape[1:], sub_ord.shape, reg_ord.shape)
    if failed.shape in [(0, 0), (0, 1)]:
        plt.figure(figsize=(4, 1))
        plt.imshow(np.array([[np.nan]]))
        plt.xticks([])
    else:
        plt.imshow(failed[reg_ord, :][:, sub_ord])
        nonad = np.where(fail_per_sub > 0)[0][sub_ord]
        plt.xticks(np.arange(len(sub_ord)), nonad + adjust[nonad], rotation=90)
    plt.title(
        f"{reg} {mat.shape[1]} {len(full_good_reg)} FF: {np.where(fail_per_sub == mat.shape[1])[0]}"
    )
    plt.xlabel(f"{failed.shape[1]} Subjects")
    plt.ylabel(f"{failed.shape[0]} Regions")
    plt.yticks([])
    plt.show()
    if sub_ord.shape[0]:
        bg = np.argsort(fail_per_sub)[-min(3, sub_ord.shape[0]) :]
        bg += adjust[bg]
        print(bg)
        bad_guys.extend(bg.tolist())
bad_selection = [a[0] for a in Counter(bad_guys).most_common(20)]
print(bad_selection)

In [ ]:
Counter(bad_guys).most_common(20)

In [ ]:
goods = np.full(245, True)
goods[187] = False
for file in glob(os.path.join(file_path, file_name)):
    findings = re.findall(r"_(\d+)\.mat", file)
    files[int(findings[0])] = file
matrices = []
weights = []
for reg in tqdm(sorted(files.keys())):
    file = files[reg]
    mat = loadmat(file)[field_name][:, :, goods]
    failed = np.sum(np.isclose(mat, 0), 0) == 400
    fail_per_reg = np.sum(failed, 1)
    keep_reg = np.full(mat.shape[1], True)
    keep_reg[np.where(fail_per_reg == 0)] = False
    matrices.append(failed[keep_reg, :])
    weights.append(np.full(np.sum(keep_reg), 1 / reg))
all_fail = np.concatenate(matrices, 0)
all_weights = np.concatenate(weights, 0)
plt.imshow(all_fail)
plt.show()

In [ ]:
all_fail = np.concatenate(matrices, 0)
all_weights = np.concatenate(weights, 0)
all_fail = (all_fail.T * all_weights).T
sub_ord = np.argsort(np.sum(all_fail, 0))
print(sub_ord[-14:], sub_ord[-14:] + adjust[sub_ord[-14:]])

In [ ]:
goods = np.full(245, True)
goods[187] = False
to_throw_binary = np.zeros(245, dtype="int64")
to_throw_binary[187] = 1
adjust = np.cumsum(to_throw_binary)[goods]
all_fail = np.concatenate(matrices, 0)
all_weights = np.concatenate(weights, 0)
all_fail_per_sub = np.sum(all_fail, 0)
full_all_failers = np.sum(all_fail_per_sub == mat.shape[1])
all_fail_per_reg = np.sum(all_fail, 1)
all_weights = all_weights[all_fail_per_reg > full_all_failers]
all_fail = (
    all_fail[:, all_fail_per_sub > 0][all_fail_per_reg > full_all_failers, :].T
    * all_weights
).T
sub_ord = np.argsort(np.sum(all_fail, 0))
reg_ord = np.argsort(np.sum(all_fail, 1))[::-1]
plt.imshow(all_fail[reg_ord, :][:, sub_ord])
plt.gca().set_aspect(0.2)
plt.xlabel(f"{all_fail.shape[1]} Subjects")
plt.ylabel(f"{all_fail.shape[0]} Regions")
plt.show()
all_fail.shape, np.sum(all_weights)

In [ ]:
best_now = all_fail[:, sub_ord[:-14]]
best_now_per_sub = np.sum(best_now, 0)
full_best_nowers = np.sum(best_now_per_sub == mat.shape[1])
best_now_per_reg = np.sum(best_now, 1)
all_weights = all_weights[best_now_per_reg > full_best_nowers]
best_now = (
    best_now[:, best_now_per_sub > 0][best_now_per_reg > full_best_nowers, :].T
    * all_weights
).T
sub_ord = np.argsort(np.sum(best_now, 0))
reg_ord = np.argsort(np.sum(best_now, 1))[::-1]
plt.imshow(best_now[reg_ord, :][:, sub_ord] > 0)
plt.gca().set_aspect(0.2)
plt.xlabel(f"{best_now.shape[1]} Subjects")
plt.ylabel(f"{best_now.shape[0]} Regions")
plt.show()
best_now.shape, np.sum(all_weights)

In [ ]:
for file in glob(os.path.join(file_path, file_name)):
    findings = re.findall(r"_(\d+)\.mat", file)
    files[int(findings[0])] = file
next_matrices = []
next_weights = []
for reg in tqdm(sorted(files.keys())):
    file = files[reg]
    mat = loadmat(file)[field_name]
    failed = np.sum(np.isclose(mat, 0), 0) == 400
    next_matrices.append(failed[:, :])
    next_weights.append(np.full(failed.shape[0], 1 / reg))

mat = loadmat("/mnt/DATA/Motion_fMRI/Datasets/Weird_manual_masks/eso/100center.mat")[
    field_name
]
failed = np.sum(np.isclose(mat, 0), 0) == 400
next_matrices.append(failed[:, :])
next_weights.append(np.full(failed.shape[0], 1 / 100))

In [ ]:
next_fail = np.concatenate(next_matrices, 0)
next_weights = np.concatenate(next_weights, 0)
next_fail = (next_fail.T * next_weights).T
sub_ord = np.argsort(np.sum(next_fail, 0))

In [ ]:
share_lost_reg = []
share_lost_wei = []
share_lost_sub = []
next_weights = []
for i, reg in tqdm(enumerate(sorted(files.keys()))):
    next_weights.append(np.full(next_matrices[i].shape[0], 1 / reg))
for i in range(0, 20):
    share_lost_sub.append(i / 245)
    kick = sub_ord[-i:] if i > 0 else []
    keep = np.full(245, True)
    keep[kick] = False
    t_reg = []
    t_wei = []
    for m, w in zip(next_matrices[:-1], next_weights[:-1]):
        failing_reg = np.sum(m[:, keep], 1) > 0
        t_reg.append(np.average(failing_reg))
        t_wei.append(np.sum(w[failing_reg]))
    share_lost_reg.append(t_reg[:])
    share_lost_wei.append(np.sum(t_wei) / 23)
total_loss = [1 - (1 - s) * (1 - r) for s, r in zip(share_lost_sub, share_lost_wei)]

In [ ]:
plt.boxplot(share_lost_reg, positions=np.arange(0, 20))
plt.scatter(np.arange(0, 20), share_lost_wei, label="Global region loss")
plt.scatter(np.arange(0, 20), share_lost_sub, label="Subject loss")
plt.scatter(np.arange(0, 20), total_loss, label="Total loss")
plt.ylim((-0.01, 0.10))
plt.legend()
plt.show()

In [ ]:
sub_ord[-3:], total_loss[3]

In [ ]:
with open("../NonLinearityResultsNew/eso245_cra_strin_SubjectMask.json") as fp:
    keep = json.load(fp)
np.where(np.logical_not(keep))[0]

In [ ]:
kick = sub_ord[-3:]
keep = np.full(245, True)
keep[kick] = False

region_masks = {}

for i, reg in tqdm(enumerate(sorted(files.keys()))):
    m = next_matrices[i]
    failing_reg = np.sum(m[:, keep], 1) > 0
    region_masks[reg] = np.logical_not(failing_reg).tolist()

In [ ]:
with open("../NonLinearityResultsNew/eso245_cra_strin_SubjectMask.json", "w") as fp:
    json.dump(keep.tolist(), fp)
with open("../NonLinearityResultsNew/eso245_cra_strin_RegionMask.json", "w") as fp:
    json.dump(region_masks, fp)

In [ ]:
for reg in tqdm(sorted(files.keys())):
    file = files[reg]
    mat = loadmat(file)[field_name]
    shaved = mat[:, :, keep][:, region_masks[reg], :]
    # print(reg, mat.shape[1:], shaved.shape[1:])
    savemat(
        f"../NonLinearityData/shaved_eso245_cra_strin/shaved_eso245_cra_strin_{reg}.mat",
        {field_name: shaved},
    )

In [ ]:
best_now = next_matrices[-1][:, keep]
sub_ord = np.argsort(np.sum(best_now, 0))
reg_ord = np.argsort(np.sum(best_now, 1))[::-1]
plt.imshow(best_now[reg_ord[:13], :][:, sub_ord[148:]] > 0)
plt.gca().set_aspect(1)
plt.xlabel(f"{best_now.shape[1]} Subjects")
plt.ylabel(f"{best_now.shape[0]} Regions")
plt.show()

In [ ]:
mat = loadmat("/mnt/DATA/Motion_fMRI/Datasets/Weird_manual_masks/eso/100center.mat")[
    field_name
]
failing_reg = np.sum(next_matrices[-1][:, keep], 1) > 0
maschera = np.logical_not(failing_reg)
shaved = mat[:, :, keep][:, maschera, :]
print(mat.shape, shaved.shape)
savemat(f"../NonLinearityData/100center.mat", {field_name: shaved})

In [ ]:
times = [
    1 / 5.23,
    1 / 2.79,
    1 / 1.3,
    1.31,
    1.79,
    2.20,
    2.65,
    2.81,
    3.2,
    3.36,
    3.79,
    4.55,
    4.55,
    5.55,
    5.35,
    6.72,
    6.55,
    7.06,
    9.75,
    9.74,
    8.61,
    9.28,
    11.51,
]
reg = [len(region_masks[reg]) for reg in region_masks]
plt.scatter(sorted(files.keys())[: len(times)], times)
plt.scatter(reg[: len(times)], times, marker="+")
plt.xlabel("Regions")
plt.ylabel("s/sub")
# plt.xscale("log")
# plt.yscale("log")
plt.show()

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from mienc.support import surrogate

x = np.zeros(400)
y = np.zeros(400)
z = np.zeros(400)

theta = (np.random.rand()-0.5)*np.pi
phi = np.random.rand()*2*np.pi
rho = 1

theta = np.cumsum(np.random.normal(0, np.pi/5, 400))
phi = np.cumsum(np.random.normal(0, np.pi/5, 400))
rho = np.exp(1 + np.cumsum(np.random.normal(0,0.05, 400)))

z = rho*np.sin(theta)
x = rho*np.cos(theta)*np.cos(phi)
y = rho*np.cos(theta)*np.sin(phi)

fig, ax = plt.subplots(1,3, gridspec_kw={'width_ratios': [1,1,1]},figsize=(15,5))
ax1 = fig.add_subplot(131,projection='3d')
ax[0].axis("off")
ax[1].axis("off")
ax[2].axis("off")

ax1.plot3D(x,y,z, ls=":")
ax1.scatter(x,y,z, c=np.arange(400))

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('"Original"')
# plt.show()
# plt.close()

vec = np.array([x,y,z]).T
x1,y1,z1 = surrogate(vec).T

# fig, ax = plt.subplots(1,3, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax2 = fig.add_subplot(132,projection='3d')
# ax[0].axis("off")
# ax[1].axis("off")

ax2.plot3D(x1,y1,z1, ls=":")
ax2.scatter(x1,y1,z1, c=np.arange(400))

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title("Cartesian surrogate")
# plt.show()
# plt.close()

vec2 = np.array([theta, phi, rho]).T
t1,p1,r1 = surrogate(vec2).T

z2 = r1*np.sin(t1)
x2 = r1*np.cos(t1)*np.cos(p1)
y2 = r1*np.cos(t1)*np.sin(p1)

# fig, ax = plt.subplots(1,3, gridspec_kw={'width_ratios': [8,0.1]},figsize=(8,6))
ax3 = fig.add_subplot(133,projection='3d')
# ax[0].axis("off")
# ax[1].axis("off")

ax3.plot3D(x2,y2,z2, ls=":")
ax3.scatter(x2,y2,z2, c=np.arange(400))

ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_zlabel('Z')
ax3.set_title("Polar surrogate")
plt.show()
# plt.close()

In [ ]:
x1.shape

In [ ]:
files = {}
for file in glob(os.path.join(file_path, file_name)):
    findings = re.findall(r"_(\d+)\.mat", file)
    files[int(findings[0])] = file
for reg in sorted(files.keys())[::-1]:
    file = files[reg]
    mat = loadmat(file)[field_name][:, :, 0]
    break

In [ ]:
plt.plot(np.fft.rfft(mat[:, :5], axis=0), marker="+")
plt.xlim((0, 80))
plt.show()

In [ ]:
bs = 1 / 2 / 400

400 / 60 * 1.5, 7 * bs, 73 * bs, mat.shape, 91.8 * 200 / 950, 35 * 48 * 64